# Phase 3: Rhetorical Roles - Legal Transformer with LoRMA

## Task
Rhetorical role labelling: every sentence of a legal document (Corporate Law and IT domains) gets one of 13 rhetorical-role labels (facts, arguments, ruling and so on). Scored with macro F1 over the labels.

Model: the pre-trained Legal Transformer is frozen; LoRMA (a multiplicative low-rank update of every attention and feed-forward weight) and a sentence classifier (a BiGRU over the sentences of a window + a linear layer) are trained. 6 epochs, snapshots at epochs 3, 4, 5 and the average of the last 2 epochs, 3 seeds for the all-documents stage.

## Input
- Phase-3 data (CL and IT train / dev / test files and the label list); downloaded from Hugging Face if not attached (needs HF_TOKEN, Internet on).
- Legal Transformer checkpoint .pt and its tokenizer (set the two paths in the Settings cell).

## Flow of the notebook (from input to output)
1. Set the file paths and training choices.
2. Write the rhetorical-role pipeline (rr_core.py) and the fine-tuning methods (rr_peft.py).
3. Write the Legal Transformer model code.
4. Import the code, load the documents, tokenize every sentence and build the training settings.
5. Build the model: frozen Legal Transformer + LoRMA + sentence classifier (with position features).
6. Train on the train documents and score on dev; then retrain on all documents and predict the test set (with optional rare-label boost and label smoothing over sentences).
7. Show the dev macro F1 of every setting.
8. Choose the decoding on the dev documents and write the top 5 submissions.
9. Zip all outputs.

## Output
- TOP5/ folder and TOP5_submissions.zip.
- All submission zips, dev scores and one zip with all outputs.

If time runs short, stop after a seed has finished and run only the TOP 5 cell and the ZIP_ALL cell.

In [1]:
# Settings: file paths and training choices
CKPT = "/kaggle/input/datasets/patelpreet2452451/legal-transformer-final-pt-p2/legal_transformer_random_moredata_best_random_90_10.pt"
TOK  = "/kaggle/input/datasets/patelpreet2452451/tokenizer-legal-transformer-p2/tokenizer_legal_transformer_random_moredata_random_90_10.json"
HF_DATASET = "Exploration-Lab/CS-781-Capstone"

EPOCHS       = 6
BS           = 4
ACCUM        = 4
W            = 512
CTX          = "window"
HEAD_LR      = 2e-3
CLASS_POWER  = 0.5
LABEL_SMOOTH = 0.05
TAUS         = [0.0]
DEV_SEEDS    = [0]
FULL_SEEDS   = [0, 1, 2]
SWA_LAST     = 2
GRU_HIDDEN   = 256
SNAP_EPOCHS  = [3, 4, 5]
VIT_LAMS     = [0.0]
POS_FEAT     = True
NEVER_PREDICT_BELOW = 50
RARE_DUP     = 2
RARE_SHARE   = 0.04
TOK_MASK     = 0.05

PEFT_LR      = 3e-4
R            = 16
ALPHA        = 32
LORA_DROPOUT = 0.05
HEAD_DROPOUT = 0.3

STAGES = ("dev", "full")
SMOKE = False
OUT = "/kaggle/working"

if SMOKE:
    EPOCHS, BS, ACCUM, W, CTX, HEAD_LR, CLASS_POWER, LABEL_SMOOTH, TAUS, DEV_SEEDS, FULL_SEEDS, SWA_LAST, GRU_HIDDEN, RARE_DUP, SNAP_EPOCHS, VIT_LAMS, POS_FEAT, NEVER_PREDICT_BELOW = 2, 2, 1, 128, "window", 2e-3, 0.5, 0.05, [0.0, 0.5], [0], [0], 1, 32, 2, [1], [0.0, 0.5], True, 5
    SMOKE_DOCS = 6
    PEFT_LR, R, ALPHA, LORA_DROPOUT, HEAD_DROPOUT = 2e-3, 4, 8, 0.05, 0.1

In [2]:
%%writefile rr_core.py
# Main rhetorical-role code: data, model, training (saved as rr_core.py)
import os, re, json, glob, math, time, copy, random, zipfile
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

REPO = "Exploration-Lab/CS-781-Capstone"
DATA_NAMES = ["CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json", "label_vocab.json", "CL_test.json", "IT_test.json"]

# It takes name and roots (folders to search) as input, searches the input folders for a file and returns a file path (or None if not found).
def find_file(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    for r in roots:
        hits = [h for h in glob.glob(os.path.join(r, "**", name), recursive=True) if "__MACOSX" not in h]
        if hits:
            return sorted(hits, key=len)[0]
    return None

# It takes dest (download folder) and repo (dataset name) as input, downloads the Phase-3 data if it is not attached and returns nothing.
def ensure_phase3_data(dest="/kaggle/working/phase3_data", repo=REPO):
    if all(find_file(n) for n in DATA_NAMES):
        print("Phase-3 files found locally:", os.path.dirname(find_file("CL_train.json"))); return
    from huggingface_hub import HfApi, hf_hub_download, login
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("Hugging Face token (read access, dataset terms accepted): ")
    login(token=tok)
    files = HfApi().list_repo_files(repo, repo_type="dataset")
    os.makedirs(dest, exist_ok=True)
    for n in DATA_NAMES:
        if find_file(n, roots=(dest,)): continue
        c = [f for f in files if f == n or f.endswith("/" + n)]
        pref = [f for f in c if any(k in f.lower() for k in ("phase_3", "phase3", "_rr", "/rr"))]
        c = pref or c
        if not c: raise RuntimeError(f"{n} not found in {repo}. Files look like: {files[:15]}")
        if len(c) > 1: print(f"  note: {n} appears {len(c)} times {c[:4]}; using {sorted(c, key=len)[0]}")
        p = hf_hub_download(repo, sorted(c, key=len)[0], repo_type="dataset", local_dir=dest)
        print("  downloaded", n, "->", p)

# It takes path (file path) as input, loads the list of labels and returns a list of labels.
def load_vocab(path):
    v = json.load(open(path, encoding="utf-8"))
    if isinstance(v, dict):
        v = [k for k, _ in sorted(v.items(), key=lambda kv: kv[1])] if all(isinstance(x, int) for x in v.values()) else list(v.values())
    v = ["None" if x is None else str(x) for x in v]
    assert len(v) == len(set(v)), "label_vocab has duplicates"
    return v

# It takes path (file path) and domain (domain name) as input, loads one data file of documents and labels and returns a dictionary of documents.
def load_split(path, domain):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict) and isinstance(raw.get("text"), list) and not any(isinstance(v, dict) for v in raw.values()): items = [raw]
    elif isinstance(raw, dict): items = [({**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    else: items = raw
    docs = {}
    for it in items:
        did = str(it["id"]); sents = [str(s) for s in it["text"]]; labs = it.get("labels")
        if labs is not None:
            labs = ["None" if l is None else str(l) for l in labs]
            assert len(labs) == len(sents), f"{path} doc {did}: {len(labs)} labels for {len(sents)} sentences"
        docs[f"{domain}|{did}"] = dict(id=did, domain=domain, sents=sents, labels=labs)
    return docs

# It takes labels (label names) as input, loads the train, dev and test documents and returns a dictionary with labels, train, dev and test.
def load_all(labels=None):
    vocab = load_vocab(find_file("label_vocab.json"))
    out = dict(labels=vocab)
    for part in ("train", "dev", "test"):
        d = {}
        for dom in ("CL", "IT"):
            p = find_file(f"{dom}_{part}.json"); assert p, f"{dom}_{part}.json not found"
            d.update(load_split(p, dom))
        out[part] = d
    for part in ("train", "dev"):
        for k, v in out[part].items():
            bad = [l for l in v["labels"] if l not in vocab]
            assert not bad, f"{k}: label(s) {sorted(set(bad))[:4]} not in label_vocab.json {vocab}"
    return out

class Enc:
    __slots__ = ("key", "id", "domain", "ids", "y", "soft")

    # It takes key, did (document id), domain (domain name), ids (token ids per sentence), y (labels) and soft (teacher probabilities) as input, stores a tokenized document and its labels and returns nothing.
    def __init__(self, key, did, domain, ids, y=None, soft=None):
        self.key, self.id, self.domain, self.ids, self.y, self.soft = key, did, domain, ids, y, soft

# It takes docs (documents), tok (tokenizer), label2i (label to id map) and max_sent_tok (max tokens per sentence) as input, tokenizes every sentence of every document and returns a dictionary of encoded documents.
def encode_docs(docs, tok, label2i, max_sent_tok):
    sep, out = tok.eos_id, {}
    for k, d in docs.items():
        ids = []
        for s in d["sents"]:
            t = list(tok.encode(s))[:max_sent_tok] or [sep]
            ids.append(t + [sep])
        y = np.array([label2i[l] for l in d["labels"]], dtype=np.int64) if d["labels"] is not None else None
        out[k] = Enc(k, d["id"], d["domain"], ids, y)
    return out

# It takes encs (encoded documents), soft_by_key (teacher probabilities per document) and keep_gold (keep true labels) as input, attaches the teacher probabilities to documents and returns a dictionary of documents with teacher probabilities.
def with_soft(encs, soft_by_key, keep_gold=False):
    out = {}
    for k, e in encs.items():
        if k not in soft_by_key: continue
        s = soft_by_key[k]; assert len(s) == len(e.ids), f"teacher probabilities for {k}: {len(s)} rows for {len(e.ids)} sentences"
        out[k] = Enc(e.key, e.id, e.domain, e.ids, e.y if keep_gold else None, s.astype(np.float32))
    return out

# It takes enc (encoded document), W (token limit), ctx (context size), rng (random generator) and step_frac (window step fraction) as input, packs sentences into windows that fit the token limit and returns a list of windows.
def windows(enc, W, ctx, rng=None, step_frac=0.5):
    n = len(enc.ids)
    if ctx == "sentence": return [(i, i + 1) for i in range(n)]
    L = [len(s) for s in enc.ids]; out, i = [], 0
    while i < n:
        j, tot = i, 0
        while j < n and (tot + L[j] <= W or j == i): tot += L[j]; j += 1
        out.append((i, j))
        if j >= n: break
        i += max(1, int((j - i) * (step_frac if rng is None else rng.uniform(0.4, 0.7))))
    return out

# It takes it (items) as input, counts the tokens in a window and returns a number of tokens.
def _tok_len(it):
    e, i, j = it
    return sum(len(e.ids[k]) for k in range(i, j))

# It takes items (list of items), pad_id (padding id), n_labels (number of labels), device, tok_mask (token masking rate) and rng (random generator) as input, builds padded tensors for one batch of windows and returns a dictionary of tensors.
def make_batch(items, pad_id, n_labels, device, tok_mask=0.0, rng=None):
    B = len(items); S = max(j - i for _, i, j in items); T = max(_tok_len(it) for it in items)
    ids = np.full((B, T), pad_id, np.int64); st = np.zeros((B, S), np.int64); en = np.zeros((B, S), np.int64)
    sm = np.zeros((B, S), bool); y = np.full((B, S), -100, np.int64)
    soft = np.zeros((B, S, n_labels), np.float32) if any(e.soft is not None for e, _, _ in items) else None
    ex = np.zeros((B, S, 4), np.float32)
    for b, (e, i, j) in enumerate(items):
        pos = 0
        for s, k in enumerate(range(i, j)):
            t = e.ids[k]; ids[b, pos:pos + len(t)] = t
            st[b, s], en[b, s], sm[b, s] = pos, pos + len(t) - 1, True
            pos += len(t)
            n = len(e.ids); ex[b, s] = (k / max(1, n - 1), float(e.domain == "IT"), math.log1p(k) / 8.0, math.log1p(n - 1 - k) / 8.0)
            if e.y is not None: y[b, s] = e.y[k]
            if soft is not None and e.soft is not None: soft[b, s] = e.soft[k]
    if tok_mask > 0 and rng is not None:
        sepm = np.zeros(ids.shape, bool)
        for b in range(B): sepm[b, en[b][sm[b]]] = True
        drop = (rng.random(ids.shape) < tok_mask) & ~sepm & (ids != pad_id)
        ids = np.where(drop, pad_id, ids)
    out = dict(ids=torch.from_numpy(ids), st=torch.from_numpy(st), en=torch.from_numpy(en), sm=torch.from_numpy(sm), y=torch.from_numpy(y), ex=torch.from_numpy(ex))
    if soft is not None: out["soft"] = torch.from_numpy(soft)
    return {k: v.to(device) for k, v in out.items()}

# It takes encs (encoded documents), cfg (settings), epoch_seed (seed for this epoch) and rare_set (rare labels) as input, groups training windows into batches of similar length and returns a list of batches.
def train_batches(encs, cfg, epoch_seed, rare_set=None):
    rng = random.Random(epoch_seed); items = []
    for e in encs.values():
        for (i, j) in windows(e, cfg["W"], cfg["ctx"], rng):
            items.append((e, i, j))
            if rare_set and cfg.get("rare_dup", 1) > 1 and e.y is not None and any(int(l) in rare_set for l in e.y[i:j]):
                items += [(e, i, j)] * (cfg["rare_dup"] - 1)
    rng.shuffle(items)
    bs, batches = cfg["bs"], []
    for g in range(0, len(items), bs * 50):
        grp = sorted(items[g:g + bs * 50], key=_tok_len)
        batches += [grp[k:k + bs] for k in range(0, len(grp), bs)]
    rng.shuffle(batches)
    return batches

class RRModel(nn.Module):

    # It takes bb (pre-trained backbone), n_labels (number of labels), ctx (context size), gru_hidden (GRU size), dropout (dropout rate), pool (pooling type) and n_extra (number of extra features) as input, builds the sentence labelling model: backbone plus sentence layers and returns nothing.
    def __init__(self, bb, n_labels, ctx="window", gru_hidden=256, dropout=0.2, pool="meanlast", n_extra=0):
        super().__init__()
        self.bb_model, self.bb, self.pool, self.use_gru = bb.model, bb, pool, (ctx != "sentence")
        d = bb.d_model; din = 2 * d if pool == "meanlast" else d
        self.drop, self.n_extra = nn.Dropout(dropout), n_extra
        pe = 0
        if n_extra:
            self.pos = nn.Sequential(nn.Linear(n_extra, 32), nn.GELU()); pe = 32
        if self.use_gru:
            self.gru = nn.GRU(din + pe, gru_hidden, batch_first=True, bidirectional=True)
            self.out = nn.Linear(2 * gru_hidden + pe, n_labels)
        else:
            self.mlp = nn.Sequential(nn.Linear(din + pe, gru_hidden), nn.GELU(), nn.Dropout(dropout)); self.out = nn.Linear(gru_hidden, n_labels)

    # It takes ids (token ids), st (sentence start positions), en (sentence end positions), sm (sentence mask) and ex (extra position features) as input, runs the model and returns a label scores for each sentence.
    def forward(self, ids, st, en, sm, ex=None):
        shim = copy.copy(self.bb); shim.model = self.bb_model
        h = shim.hidden(ids).float()
        d = h.size(-1)
        cs = F.pad(h.cumsum(1), (0, 0, 1, 0))
        gat = lambda idx: torch.gather(cs, 1, idx.unsqueeze(-1).expand(-1, -1, d))
        mean = (gat(en) - gat(st)) / (en - st).clamp(min=1).unsqueeze(-1).float()
        last = torch.gather(h, 1, en.unsqueeze(-1).expand(-1, -1, d))
        x = mean if self.pool == "mean" else last if self.pool == "last" else torch.cat([mean, last], -1)
        x = self.drop(x)
        pe = self.pos(ex.float()) if self.n_extra else None
        if pe is not None: x = torch.cat([x, pe], -1)
        if self.use_gru:
            lens = sm.sum(1).clamp(min=1).cpu()
            y, _ = self.gru(nn.utils.rnn.pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False))
            x, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=sm.size(1))
            x = self.drop(x)
            return self.out(torch.cat([x, pe], -1) if pe is not None else x)
        return self.out(self.mlp(x))

# It takes model as input, counts trainable and total parameters and returns (trainable parameters, total parameters).
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())

# It takes model, head_lr (head learning rate), peft_lr (learning rate for the adapter weights) and wd (weight decay) as input, sets learning rates for the head and the other trainable weights and returns a list of parameter groups with learning rates.
def make_groups(model, head_lr, peft_lr, wd=0.01):
    head = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("bb_model.")]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("bb_model.")]
    g = [dict(params=head, lr=head_lr, weight_decay=wd)]
    if rest: g.append(dict(params=rest, lr=peft_lr, weight_decay=wd))
    return g

# It takes gold (true labels), pred (predictions) and n_labels (number of labels) as input, computes macro F1 and other scores and returns a dictionary with macro F1 and other scores.
def scores(gold, pred, n_labels):
    cm = np.zeros((n_labels, n_labels), np.int64); np.add.at(cm, (gold, pred), 1)
    tp = np.diag(cm).astype(float); sup, pc = cm.sum(1), cm.sum(0)
    P = np.divide(tp, pc, out=np.zeros(n_labels), where=pc > 0); R = np.divide(tp, sup, out=np.zeros(n_labels), where=sup > 0)
    F1 = np.divide(2 * P * R, P + R, out=np.zeros(n_labels), where=(P + R) > 0)
    pres = (sup > 0) | (pc > 0)
    return dict(macro_f1=float(F1[pres].mean()), macro_p=float(P[pres].mean()), macro_r=float(R[pres].mean()), macro_f1_all13=float(F1.mean()),
                per_label=dict(P=P.tolist(), R=R.tolist(), F1=F1.tolist(), support=sup.tolist()), confusion=cm.tolist())

# It takes probs (probabilities), prior (label frequencies), tau (rare-label boost) and banned (labels never predicted) as input, boosts rare labels by adjusting the probabilities and returns a adjusted probabilities.
def adjust(probs, prior, tau, banned=()):
    p = probs if tau == 0 else probs / np.power(prior[None, :], tau)
    if len(banned): p = p.copy(); p[:, list(banned)] = 0.0
    return p / np.maximum(p.sum(1, keepdims=True), 1e-12)

# It takes encs (encoded documents) and n (number) as input, learns how likely each label follows another and returns a matrix of log transition probabilities.
def transitions(encs, n):
    T = np.ones((n, n))
    for e in encs.values():
        if e.y is not None and len(e.y) > 1: np.add.at(T, (e.y[:-1], e.y[1:]), 1)
    return np.log(T / T.sum(1, keepdims=True))

# It takes logp (log probabilities), logT (label transition scores) and lam (strength) as input, finds the best label sequence for the sentences and returns the best label id for each position.
def viterbi(logp, logT, lam):
    n, K = logp.shape
    if not lam or n < 2: return logp.argmax(1)
    dp, bp = logp[0].copy(), np.zeros((n, K), np.int64)
    for t in range(1, n):
        cand = dp[:, None] + lam * logT; bp[t] = cand.argmax(0); dp = cand.max(0) + logp[t]
    y = np.zeros(n, np.int64); y[-1] = int(dp.argmax())
    for t in range(n - 1, 0, -1): y[t - 1] = bp[t, y[t]]
    return y

# It takes p (probabilities), prior (label frequencies), tau (rare-label boost), logT (label transition scores), lam (strength) and banned (labels never predicted) as input, picks a label for every sentence and returns a label id for every sentence.
def decode(p, prior, tau, logT=None, lam=0.0, banned=()):
    q = adjust(p, prior, tau, banned)
    return q.argmax(1) if not lam else viterbi(np.log(q + 1e-9), logT, lam)

# It takes encs (encoded documents), probs (probabilities), n_labels (number of labels), prior (label frequencies), tau (rare-label boost), by_domain (also score per domain), logT (label transition scores), lam (strength) and banned (labels never predicted) as input, scores predictions on a split, also per domain and returns a dictionary of scores.
def score_split(encs, probs, n_labels, prior=None, tau=0.0, by_domain=True, logT=None, lam=0.0, banned=()):
    # It takes keys as input, scores a group of documents and returns a dictionary of scores.
    def run(keys):
        g = np.concatenate([encs[k].y for k in keys]); p = np.concatenate([decode(probs[k], prior, tau, logT, lam, banned) for k in keys]); return scores(g, p, n_labels)
    keys = [k for k in encs if encs[k].y is not None and k in probs]
    out = {"all": run(keys)}
    if by_domain:
        for dom in sorted({encs[k].domain for k in keys}):
            ks = [k for k in keys if encs[k].domain == dom]
            if ks: out[dom] = run(ks)
    return out

# It takes encs (encoded documents) and n_labels (number of labels) as input, counts how many sentences have each label and returns a count per label.
def label_counts(encs, n_labels):
    c = np.zeros(n_labels)
    for e in encs.values():
        if e.y is not None: c += np.bincount(e.y[e.y >= 0], minlength=n_labels)
    return c

# It takes encs (encoded documents), n_labels (number of labels), power, cap (maximum value) and min_count (minimum count) as input, gives rare labels a higher loss weight and returns (weight per label, label frequencies).
def class_weights(encs, n_labels, power=0.5, cap=4.0, min_count=50):
    c = np.maximum(label_counts(encs, n_labels), max(1, min_count)); w = (c.sum() / (n_labels * c)) ** power
    return torch.tensor(np.minimum(w / w.mean(), cap), dtype=torch.float32), c / c.sum()

# It takes e (error) as input, checks whether an error is a GPU out-of-memory error and returns True or False.
def _is_oom(e): return "out of memory" in str(e).lower()

# It takes model, encs (encoded documents), cfg (settings), device, n_labels (number of labels), pad_id (padding id) and bs (batch size) as input, predicts label probabilities for every sentence and returns probabilities for every document.
@torch.no_grad()
def predict_probs(model, encs, cfg, device, n_labels, pad_id, bs=None):
    model.eval(); bs = bs or cfg["bs"] * 2
    use_amp = device == "cuda"
    items = [(e, i, j) for e in encs.values() for (i, j) in windows(e, cfg["W"], cfg["ctx"], None, step_frac=cfg.get("tta_step", 0.5))]
    order = sorted(range(len(items)), key=lambda k: _tok_len(items[k]))
    acc = {k: np.zeros((len(e.ids), n_labels), np.float32) for k, e in encs.items()}; wsum = {k: np.zeros(len(e.ids), np.float32) for k, e in encs.items()}
    dmodel = nn.DataParallel(model) if (device == "cuda" and torch.cuda.device_count() > 1) else model
    p = 0
    while p < len(order):
        part = [items[k] for k in order[p:p + bs]]
        b = make_batch(part, pad_id, n_labels, device)
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                pr = torch.softmax(dmodel(b["ids"], b["st"], b["en"], b["sm"], b["ex"]).float(), -1).cpu().numpy()
        except RuntimeError as ex:
            if _is_oom(ex) and bs > 1: torch.cuda.empty_cache(); bs = max(1, bs // 2); continue
            raise
        for r, (e, i, j) in enumerate(part):
            for s, k in enumerate(range(i, j)):
                pos = (s + 0.5) / (j - i); w = 1.0 - 0.8 * abs(2 * pos - 1)
                acc[e.key][k] += w * pr[r, s]; wsum[e.key][k] += w
        p += len(part)
    return {k: acc[k] / np.maximum(wsum[k], 1e-9)[:, None] for k in acc}

PROGRESS = dict(total=0.0, done=0.0, skipped=0.0, t0=None)

# It takes sec (seconds) as input, formats seconds as hours:minutes:seconds and returns time as text.
def _hms(sec):
    sec = int(max(0, sec)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

# It shows overall progress and time left and returns progress text.
def progress_line():
    P = PROGRESS
    if not P["total"] or not P["t0"] or P["done"] <= 0: return ""
    el = time.time() - P["t0"]; rate = P["done"] / max(1e-9, el); left = max(0.0, P["total"] - P["done"] - P["skipped"])
    return f"[overall {100 * min(1.0, (P['done'] + P['skipped']) / P['total']):.0f}% | elapsed {_hms(el)} | ETA {_hms(left / rate)}]"

# It takes model as input, copies the trainable weights and returns a dictionary of trainable weights.
def _tstate(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}

# It takes model, train_encs (encoded training documents), val_encs (encoded validation documents), cfg (settings), groups (parameter groups with learning rates), device, n_labels (number of labels), pad_id (padding id), log (print function) and ckpt_path (checkpoint path) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(model, train_encs, val_encs, cfg, groups, device, n_labels, pad_id, log=print, ckpt_path=None):
    use_amp = device == "cuda"
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), weight_decay=cfg.get("wd", 0.01)); base_lr = [g["lr"] for g in groups]
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw, prior = class_weights(train_encs, n_labels, cfg.get("class_power", 0.5), min_count=cfg.get("min_count", 50)); cw = cw.to(device)
    rare = {i for i, p in enumerate(prior) if p < cfg.get("rare_share", 0.03)}
    tok_mask = cfg.get("tok_mask", 0.0)
    dmodel = nn.DataParallel(model) if (device == "cuda" and torch.cuda.device_count() > 1) else model
    accum, E = cfg["accum"], cfg["epochs"]
    n_b = len(train_batches(train_encs, cfg, 0, rare)); steps_per_epoch = max(1, n_b // accum); total = steps_per_epoch * E
    warm = max(1, int(cfg.get("warmup", 0.06) * total))
    log(f"  micro-batches/epoch {n_b} (bs {cfg['bs']} x accum {accum}), optimizer steps/epoch {steps_per_epoch}, total {total}; rare labels {sorted(rare)}")
    kd_a, kd_T = cfg.get("kd_alpha", 0.0), cfg.get("kd_T", 2.0)
    train_params = [p for g in groups for p in g["params"]]
    step, ep0, best, best_ep, hist, best_state, avg_state, avg_n = 0, 0, -1.0, -1, [], None, None, 0
    snaps, snap_eps = {}, set(cfg.get("snap_epochs") or [])
    if ckpt_path and os.path.exists(ckpt_path):
        try:
            ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["state"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
            step, ep0, best, best_ep, hist = ck["step"], ck["ep"], ck["best"], ck["best_ep"], ck["hist"]
            best_state, avg_state, avg_n = ck["best_state"], ck["avg_state"], ck["avg_n"]; snaps = ck.get("snaps", {})
            PROGRESS["skipped"] += ep0 * len(train_encs); log(f"  resumed from {os.path.basename(ckpt_path)} at epoch {ep0 + 1}/{E}"); del ck
        except Exception as ex:
            log(f"  could not resume ({str(ex)[:100]}) -> training from scratch"); step, ep0, best, best_ep, hist, best_state, avg_state, avg_n = 0, 0, -1.0, -1, [], None, None, 0
    t_save, bs_now = time.time(), cfg["bs"]
    for ep in range(ep0, E):
        model.train(); t0 = time.time(); run, nrun = torch.zeros((), device=device), 0
        if PROGRESS["t0"] is None: PROGRESS["t0"] = t0
        batches = train_batches(train_encs, cfg, cfg.get("seed", 0) * 1000 + ep, rare); arng = np.random.default_rng(cfg.get("seed", 0) * 1000 + ep + 7)
        ep_steps = max(1, len(batches) // accum)
        try:
            from tqdm.auto import tqdm as _tqdm
            bar = _tqdm(total=ep_steps, desc=f"epoch {ep + 1}/{E}", mininterval=20, leave=False, dynamic_ncols=True)
        except Exception:
            bar = None
        for s0 in range(0, ep_steps * accum, accum):
            lr_f = (step + 1) / warm if step < warm else max(0.02, 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm)))))
            for g, b0 in zip(opt.param_groups, base_lr): g["lr"] = b0 * lr_f
            if not batches[s0:s0 + accum]: break
            for part in batches[s0:s0 + accum]:
                b = make_batch(part, pad_id, n_labels, device, tok_mask, arng)
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        lg = dmodel(b["ids"], b["st"], b["en"], b["sm"], b["ex"]).float()
                    ym = b["y"] != -100
                    ce = F.cross_entropy(lg[ym], b["y"][ym], weight=cw, label_smoothing=cfg.get("label_smooth", 0.0)) if bool(ym.any()) else lg.sum() * 0.0
                    loss = ce
                    if kd_a > 0 and "soft" in b:
                        has = (b["soft"].sum(-1) > 0) & b["sm"]
                        if bool(has.any()):
                            tgt = b["soft"][has].clamp_min(1e-9) ** (1 / kd_T); tgt = tgt / tgt.sum(-1, keepdim=True)
                            kd = F.kl_div(F.log_softmax(lg[has] / kd_T, -1), tgt, reduction="batchmean") * kd_T * kd_T
                            frac = has.float().sum() / (ym | has).float().sum().clamp(min=1)
                            loss = (1 - kd_a * frac) * ce + kd_a * frac * kd
                    scaler.scale(loss / accum).backward(); run += ce.detach().float(); nrun += 1
                except RuntimeError as ex:
                    if _is_oom(ex):
                        raise RuntimeError("CUDA out of memory: lower cfg['bs'] (windows per micro-batch) or cfg['W'], and raise cfg['accum'] to keep the same effective batch") from ex
                    raise
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(train_params, cfg.get("clip", 1.0))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            if bar is not None:
                bar.update(1)
                if step % 25 == 0: bar.set_postfix(ce=f"{float(run) / max(1, nrun):.3f}")
        if bar is not None: bar.close()
        PROGRESS["done"] += len(train_encs)
        rec = dict(epoch=ep + 1, train_ce=float(run) / max(1, nrun), secs=time.time() - t0)
        if val_encs:
            pr = predict_probs(model, val_encs, cfg, device, n_labels, pad_id); sc = score_split(val_encs, pr, n_labels)
            rec.update(val_macro_F1=sc["all"]["macro_f1"], **{f"val_{d}": sc[d]["macro_f1"] for d in sc if d != "all"})
            if rec["val_macro_F1"] > best: best, best_ep, best_state = rec["val_macro_F1"], ep + 1, _tstate(model)
        if cfg.get("swa_last", 0) and ep + 1 > E - cfg["swa_last"]:
            cur = _tstate(model)
            if avg_state is None: avg_state, avg_n = {k: v.float() for k, v in cur.items()}, 1
            else:
                avg_n += 1
                for k, v in cur.items(): avg_state[k] += (v.float() - avg_state[k]) / avg_n
        if ep + 1 in snap_eps: snaps[ep + 1] = _tstate(model)
        hist.append(rec)
        log("  " + "  ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()) +
            f"  | epoch {ep + 1}/{E}, run ETA {_hms((time.time() - t0) * (E - ep - 1))}  " + progress_line())
        if ckpt_path and ep + 1 < E and time.time() - t_save >= cfg.get("save_every", 1800):
            torch.save(dict(state=_tstate(model), opt=opt.state_dict(), scaler=scaler.state_dict(), step=step, ep=ep + 1, best=best, best_ep=best_ep,
                            hist=hist, best_state=best_state, avg_state=avg_state, avg_n=avg_n, snaps=snaps), ckpt_path + ".tmp")
            os.replace(ckpt_path + ".tmp", ckpt_path); t_save = time.time(); log(f"  [saved resumable checkpoint after epoch {ep + 1}]")
    last_state = _tstate(model)
    if best_state is None: best_state, best_ep = last_state, len(hist)
    if avg_state is not None: avg_state = {k: v.to(last_state[k].dtype) if last_state[k].is_floating_point() else last_state[k] for k, v in avg_state.items()}
    if ckpt_path and os.path.exists(ckpt_path): os.remove(ckpt_path)
    return hist, best_ep, best_state, last_state, avg_state, snaps

# It takes model and state (weights) as input, loads saved weights into the model and returns nothing.
def load_state(model, state):
    miss = model.load_state_dict(state, strict=False); assert not miss.unexpected_keys, miss.unexpected_keys

# It takes s (random seed) as input, sets random seeds so runs are repeatable and returns nothing.
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# It takes path (file path) and probs (probabilities) as input, saves predicted probabilities to a file and returns nothing.
def save_probs(path, probs):
    keys = sorted(probs); np.savez_compressed(path, keys=np.array(keys), **{f"p{i}": probs[k].astype(np.float16) for i, k in enumerate(keys)})

# It takes path (file path) as input, loads saved probabilities from a file and returns probabilities per document.
def load_probs(path):
    z = np.load(path, allow_pickle=True); return {str(k): z[f"p{i}"].astype(np.float32) for i, k in enumerate(z["keys"])}

# It takes plist (list of probabilities) as input, averages several probability dictionaries and returns averaged probabilities.
def avg_probs(plist):
    return {k: sum(p[k] for p in plist) / len(plist) for k in plist[0]}

# It takes test_encs (encoded test documents), probs (probabilities), labels (label names), prior (label frequencies), tau (rare-label boost), logT (label transition scores), lam (strength) and banned (labels never predicted) as input, makes the label list for every test document and returns a dictionary from document id to labels.
def predictions_for(test_encs, probs, labels, prior, tau, logT=None, lam=0.0, banned=()):
    pred = {}
    for k, e in test_encs.items():
        assert e.id not in pred, (f"document id {e.id!r} occurs in BOTH CL and IT test files - one prediction.json cannot hold both. "
                                  "Look at the evaluation script of the course to see how the two domains must be combined, then change this function.")
        lab = decode(probs[k], prior, tau, logT, lam, banned); pred[e.id] = [labels[int(i)] for i in lab]
    return pred

# It takes pred (predictions), test_docs (test documents) and labels (label names) as input, checks the predictions match the test documents and returns True (stops with an error if something is wrong).
def check_prediction(pred, test_docs, labels):
    ids_t = {d["id"] for d in test_docs.values()}
    assert ids_t == set(pred), f"id mismatch: missing {sorted(ids_t - set(pred))[:3]} unknown {sorted(set(pred) - ids_t)[:3]}"
    lens = {d["id"]: len(d["sents"]) for d in test_docs.values()}; vs = set(labels)
    for k, v in pred.items():
        assert len(v) == lens[k], f"{k}: {len(v)} labels for {lens[k]} sentences"
        assert all(l in vs for l in v), f"{k}: label not in label_vocab"
    return True

# It takes pred (predictions) and out_zip (zip path) as input, writes predictions into the submission zip and returns a path of the zip.
def write_submission(pred, out_zip):
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z: z.writestr("prediction.json", json.dumps(pred, ensure_ascii=False))
    with zipfile.ZipFile(out_zip) as z: assert z.namelist() == ["prediction.json"]
    return out_zip

# It takes cm (confusion matrix), labels (label names), path (file path) and title as input, plots the confusion matrix and returns nothing.
def plot_confusion(cm, labels, path, title=""):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    cm = np.array(cm, float); R = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(9, 8)); im = ax.imshow(R, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("gold (row-normalised)"); ax.set_title(title); fig.colorbar(im); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

# It takes name, build (function that builds the model), cfg (settings), data (dataset splits), out_dir (output folder), work_dir (work folder), resume_dirs (folders of earlier runs), log (print function), soft_train (teacher-labelled training documents), taus (rare-label boost values), dev_seeds (seeds for the dev stage), full_seeds (seeds for the full stage) and stages (stages to run) as input, runs one full experiment: train, evaluate and save outputs and returns a results summary.
def run_experiment(name, build, cfg, data, out_dir, work_dir, resume_dirs=(), log=print, soft_train=None, taus=(0.0, 0.5, 1.0),
                   dev_seeds=(0,), full_seeds=(0,), stages=("dev", "full")):
    os.makedirs(out_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)
    labels, K, pad_id = data["labels"], len(data["labels"]), data["pad_id"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _, prior = class_weights({**data["train"], **data["dev"]}, K, min_count=cfg.get("min_count", 50))
    summary = dict(name=name, labels=labels, cfg={k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool, list, tuple, type(None)))}, stages={})
    n_docs = {"dev": len(data["train"]), "full": len(data["train"]) + len(data["dev"])}
    PROGRESS.update(total=cfg["epochs"] * (("dev" in stages) * n_docs["dev"] * len(dev_seeds) + ("full" in stages) * n_docs["full"] * len(full_seeds)), done=0.0, skipped=0.0, t0=None)

    # It takes fn (file name) as input, finds a saved file from this or an earlier run and returns a file path (or None).
    def have(fn):
        for d in (work_dir,) + tuple(resume_dirs):
            p = os.path.join(d, fn)
            if os.path.exists(p): return p
        return None

    cnt_all = label_counts({**data["train"], **data["dev"]}, K); cnt_tr = label_counts(data["train"], K)
    thr = cfg.get("never_predict_below", 0)
    banned = {"dev": tuple(int(i) for i in np.where(cnt_tr < thr)[0]), "full": tuple(int(i) for i in np.where(cnt_all < thr)[0])}
    log("labels that are never predicted (fewer than %s training sentences): %s" % (thr, [labels[i] for i in banned["full"]] or "none"))
    lams = list(cfg.get("vit_lams") or [0.0]); vs = lambda l: f"_vit{l:g}" if l else ""
    # It takes stage (stage name), kind (type), probs_test (test probabilities) and logT (label transition scores) as input, writes submission files for the given predictions and returns nothing.
    def emit(stage, kind, probs_test, logT):
        for t in taus:
            for l in lams:
                pred = predictions_for(data["test"], probs_test, labels, prior, t, logT, l, banned[stage]); check_prediction(pred, data["raw"]["test"], labels)
                fn = f"{name}__{stage}_{kind}__tau{t:g}{vs(l)}_submission.zip"; write_submission(pred, os.path.join(out_dir, fn)); log(f"    wrote {fn}")

    for stage in [s for s in ("dev", "full") if s in stages]:
        seeds = dev_seeds if stage == "dev" else full_seeds
        log(f"\n===== {name}: stage '{stage}' ({'train -> dev' if stage == 'dev' else 'train + dev, all documents'}), seeds {list(seeds)} =====")
        from collections import defaultdict
        per_kind, per_kind_dev, hist_all = defaultdict(list), defaultdict(list), []
        # It uses values from the outer function, averages the predictions of saved epochs and returns nothing.
        def add_snapens():
            eps = sorted((k for k in per_kind if re.fullmatch(r"ep\d+", k)), key=lambda k: int(k[2:]))
            if eps and per_kind.get("final"):
                ks = eps + ["final"]; n = len(per_kind["final"])
                per_kind["snapens"] = [avg_probs([per_kind[k][i] for k in ks]) for i in range(n)]
                if stage == "dev": per_kind_dev["snapens"] = [avg_probs([per_kind_dev[k][i] for k in ks]) for i in range(n)]
        # It uses values from the outer function, saves the averaged predictions so far and returns nothing.
        def dump_partial():
            add_snapens()
            for kind in per_kind:
                if per_kind[kind]:
                    save_probs(os.path.join(out_dir, f"{name}_{stage}_{kind}_test_probs.npz"), avg_probs(per_kind[kind]))
                    if stage == "dev" and per_kind_dev.get(kind): save_probs(os.path.join(out_dir, f"{name}_dev_{kind}_dev_probs.npz"), avg_probs(per_kind_dev[kind]))
        logT = transitions({**data["train"], **(data["dev"] if stage == "full" else {})}, K)
        for seed in seeds:
            fn = f"{name}_{stage}_s{seed}.npz"; got = have(fn)
            if got:
                z = np.load(got, allow_pickle=True); log(f"  seed {seed}: found finished run {fn} -> skipped"); PROGRESS["skipped"] += cfg["epochs"] * n_docs[stage]
                for kind in [f[:-10] for f in z.files if f.endswith("_test_keys")]:
                    if True:
                        per_kind[kind].append({str(k): z[f"{kind}_test_{i}"].astype(np.float32) for i, k in enumerate(z[f"{kind}_test_keys"])})
                        if stage == "dev": per_kind_dev[kind].append({str(k): z[f"{kind}_dev_{i}"].astype(np.float32) for i, k in enumerate(z[f"{kind}_dev_keys"])})
                hist_all.append(json.loads(str(z["hist"]))); continue
            seed_all(seed); model, groups, info = build(seed); model.to(device)
            tr, al = count_params(model); log(f"  seed {seed}: trainable parameters {tr:,} of {al:,} ({100 * tr / al:.2f} %)  {info or ''}")
            train = dict(data["train"]);
            if stage == "full": train.update(data["dev"])
            if soft_train: train.update(with_soft(train, soft_train[stage], keep_gold=True))
            c = dict(cfg, seed=seed)
            hist, best_ep, bst, lst, ast, snaps = fit(model, train, data["dev"] if stage == "dev" else None, c, groups, device, K, pad_id, log, os.path.join(work_dir, f"{name}_{stage}_s{seed}.resume.pt"))
            store, kinds = {}, ([("best", bst), ("final", lst), ("avg", ast)] if stage == "dev" else [("final", lst), ("avg", ast)])
            kinds += [(f"ep{e}", st) for e, st in sorted(snaps.items()) if e != cfg["epochs"]]
            kinds = [kv for kv in kinds if kv[1] is not None]
            for kind, st in kinds:
                if st is None: continue
                load_state(model, st)
                pt = predict_probs(model, data["test"], c, device, K, pad_id); per_kind[kind].append(pt)
                store[f"{kind}_test_keys"] = np.array(sorted(pt)); store.update({f"{kind}_test_{i}": pt[k].astype(np.float16) for i, k in enumerate(sorted(pt))})
                if stage == "dev":
                    pd_ = predict_probs(model, data["dev"], c, device, K, pad_id); per_kind_dev[kind].append(pd_)
                    store[f"{kind}_dev_keys"] = np.array(sorted(pd_)); store.update({f"{kind}_dev_{i}": pd_[k].astype(np.float16) for i, k in enumerate(sorted(pd_))})
            if cfg.get("dump_train") and lst is not None:
                load_state(model, lst); ptr = predict_probs(model, train, c, device, K, pad_id); save_probs(os.path.join(out_dir, f"{name}_{stage}_s{seed}_train_soft.npz"), ptr)
            np.savez_compressed(os.path.join(work_dir, fn), hist=json.dumps(hist), best_ep=best_ep, **store); hist_all.append(hist)
            dump_partial(); log(f"  seed {seed} finished: probabilities of {len(hist_all)} seed(s) so far are saved")
            del model
            if device == "cuda": torch.cuda.empty_cache()
        add_snapens()
        info_stage = dict(history=hist_all, scores={})
        for kind in sorted(per_kind, key=lambda k: (k not in ("best", "final", "avg"), k)):
            if not per_kind[kind]: continue
            pt = avg_probs(per_kind[kind]); save_probs(os.path.join(out_dir, f"{name}_{stage}_{kind}_test_probs.npz"), pt)
            if stage == "dev":
                pdv = avg_probs(per_kind_dev[kind]); save_probs(os.path.join(out_dir, f"{name}_dev_{kind}_dev_probs.npz"), pdv)
                for t in taus:
                  for l in lams:
                    sc = score_split(data["dev"], pdv, K, prior, t, logT=logT, lam=l, banned=banned[stage]); info_stage["scores"][f"{kind}_tau{t:g}{vs(l)}"] = {d: {k: v for k, v in s.items()} for d, s in sc.items()}
                    log(f"  dev Macro-F1 [{kind:5s} tau {t:g}{' viterbi ' + format(l, 'g') if l else ''}]  all {sc['all']['macro_f1']:.4f}  " + "  ".join(f"{d} {sc[d]['macro_f1']:.4f}" for d in sc if d != 'all') + f"  (all 13 labels: {sc['all']['macro_f1_all13']:.4f})")
                sc0 = info_stage["scores"][f"{kind}_tau{taus[0]:g}"]["all"]
                log("    per label (first tau, no viterbi):  " + "  ".join(f"{l} F1 {f:.2f} (n={int(n)})" for l, f, n in zip(labels, sc0["per_label"]["F1"], sc0["per_label"]["support"])))
                plot_confusion(sc0["confusion"], labels, os.path.join(out_dir, f"{name}_dev_{kind}_confusion.png"), f"{name} dev {kind}")
            if cfg.get("emit_zips", True): emit(stage, kind, pt, logT)
        summary["stages"][stage] = info_stage
        json.dump(summary, open(os.path.join(out_dir, f"{name}_summary.json"), "w"), indent=1)
    return summary

Writing rr_core.py


In [3]:
%%writefile rr_peft.py
# Fine-tuning methods: LoRA, LoRMA, adapters (saved as rr_peft.py)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# It takes model and name as input, gets a sub-layer of the model by its name and returns the sub-layer.
def _get(model, name):
    m = model
    for p in name.split("."):
        m = getattr(m, p)
    return m

# It takes model, name and new (new layer) as input, replaces a sub-layer of the model by its name and returns nothing.
def _set(model, name, new):
    parts = name.split(".")
    parent = _get(model, ".".join(parts[:-1])) if len(parts) > 1 else model
    setattr(parent, parts[-1], new)

class LoRALinear(nn.Module):
    # It takes base (base linear layer), r (rank), alpha (scale of the low-rank update), dropout (dropout rate) and rs (use rsLoRA scaling) as input, creates the small LoRA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the linear layer plus the LoRA update and returns the output of the updated layer.
    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))

class LoRMALinear(nn.Module):

    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.s = base, alpha / r
        o = base.out_features
        self.A = nn.Parameter(torch.empty(r, o)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(o, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

# It takes model as input, freezes all weights of the model and returns nothing.
def freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)

# It takes bb (pre-trained backbone), kind (type), names (layer names), r (rank), alpha (scale of the low-rank update) and dropout (dropout rate) as input, freezes the model and wraps chosen linear layers with LoRA or LoRMA and returns a number of layers wrapped.
def apply_linear_wrappers(bb, kind, names, r=8, alpha=16, dropout=0.05):
    freeze(bb.model)
    n = 0
    for name in names:
        base = _get(bb.model, name)
        if not isinstance(base, nn.Linear):
            continue
        new = LoRMALinear(base, r, alpha) if kind == "lorma" else LoRALinear(base, r, alpha, dropout, rs=(kind == "rslora"))
        _set(bb.model, name, new); n += 1
    assert n > 0, "no Linear layers wrapped - check the name list"
    return n

class Adapter(nn.Module):
    # It takes d (model size), bottleneck (adapter size) and dropout (dropout rate) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up, self.drop = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    # It takes h (hidden vectors) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(self, h):
        return h + self.up(self.drop(F.gelu(self.down(h))))

# It takes bb (pre-trained backbone) as input, returns the list of model blocks.
def get_blocks(bb):
    if hasattr(bb, "block_modules"):
        return list(bb.block_modules())
    return list(bb.model.blocks)

# It takes bb (pre-trained backbone), bottleneck (adapter size), dropout (dropout rate) and every (how often) as input, freezes the model and adds an adapter after the blocks and returns a number of adapters added.
def add_block_adapters(bb, bottleneck=64, dropout=0.1, every=1):
    freeze(bb.model)
    blocks = get_blocks(bb)
    n = 0
    for i, blk in enumerate(blocks):
        if i % every: continue
        blk.add_module("ner_adapter", Adapter(bb.d_model, bottleneck, dropout))

        # It takes mod (module), inp (layer input) and out (block output) as input, adds the adapter output to the block output and returns block output plus adapter output.
        def hook(mod, inp, out):
            ad = mod.ner_adapter
            if isinstance(out, tuple):
                return (ad(out[0]),) + tuple(out[1:])
            return ad(out)
        blk.register_forward_hook(hook); n += 1
    return n

# It takes bb (pre-trained backbone), last_k_blocks (number of top blocks to train), train_norms (also train norm layers) and train_embeddings (also train embeddings) as input, freezes the model except the last few blocks and returns a number of trainable parameters.
def partial_finetune(bb, last_k_blocks=4, train_norms=True, train_embeddings=False):
    freeze(bb.model)
    blocks = get_blocks(bb)
    for blk in blocks[-last_k_blocks:]:
        for p in blk.parameters(): p.requires_grad_(True)
    if train_norms:
        for n, p in bb.model.named_parameters():
            if "norm_f" in n or n.startswith("norm.") or n.startswith("ln_f"):
                p.requires_grad_(True)
    if train_embeddings:
        for n, p in bb.model.named_parameters():
            if "embed" in n or "tok" in n.split(".")[0]: p.requires_grad_(True)
    return sum(p.numel() for p in bb.model.parameters() if p.requires_grad)

# It takes bb (pre-trained backbone), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, gives lower layers a smaller learning rate than top layers and returns a list of parameter groups with learning rates.
def layerwise_lr_groups(bb, lr_top, decay=0.85, head_params=(), head_lr=1e-3, wd=0.01):
    blocks = get_blocks(bb)
    n = len(blocks)
    seen, groups = set(), []
    for i, blk in enumerate(blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        for p in ps: seen.add(id(p))
        groups.append(dict(params=ps, lr=lr_top * decay ** (n - 1 - i), weight_decay=wd))
    rest = [p for p in bb.model.parameters() if p.requires_grad and id(p) not in seen]
    groups.append(dict(params=rest, lr=lr_top * decay ** n, weight_decay=wd))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return [g for g in groups if g["params"]]

Writing rr_peft.py


In [4]:
%%writefile arch_legal_transformer.py
# Legal Transformer model code (saved as arch_legal_transformer.py)
import json
import math
import os
import re as _re
import shutil
import tempfile
import time as _time
import zipfile
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "legal_transformer"
TRAINED_SEQ_TOKENS = 640
NOTEBOOK_ROPE_LEN = TRAINED_SEQ_TOKENS + 8

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}

_PRETOK_RE = _re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)

            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]

        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None

        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]

        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

# It takes head_dim (size of one head), max_len (max length), device and base (rotary base value) as input, pre-computes the sine and cosine tables for rotary positions and returns (cos table, sin table).
def build_rope_cache(head_dim, max_len, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

# It takes x (input tensor) as input, rotates half of the vector, used by rotary positions and returns rotated tensor.
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

# It takes q (queries), k (keys), cos (rotary cos table), sin (rotary sin table) and pos_offset (position offset) as input, adds rotary position information to queries and keys and returns (rotated queries, rotated keys).
def apply_rope(q, k, cos, sin, pos_offset):
    T = q.size(2)
    cos_t = cos[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    sin_t = sin[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    q_rot = q * cos_t + rotate_half(q) * sin_t
    k_rot = k * cos_t + rotate_half(k) * sin_t
    return q_rot, k_rot

USE_SDPA = True

class CausalSelfAttention(nn.Module):

    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates the attention layer and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, dim)
        self.wk = nn.Linear(dim, dim)
        self.wv = nn.Linear(dim, dim)
        self.wo = nn.Linear(dim, dim)
        self.attn_drop_p = dropout
        self.resid_drop = nn.Dropout(dropout)
        self.bidirectional = False
        self.key_pad_mask = None

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs attention where each token looks only at earlier tokens and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin, pos_offset)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)

        Tq, Tk = q.size(2), k.size(2)
        q_idx = torch.arange(Tk - Tq, Tk, device=x.device)
        k_idx = torch.arange(Tk, device=x.device)
        causal_mask = k_idx.unsqueeze(0) > q_idx.unsqueeze(1)
        if self.bidirectional:
            causal_mask = torch.zeros_like(causal_mask)

        if USE_SDPA:
            allowed = ~causal_mask
            if self.key_pad_mask is not None:
                allowed = allowed[None, None] & self.key_pad_mask[:, None, None, :]
                allowed = allowed | (k_idx.unsqueeze(0) == q_idx.unsqueeze(1))[None, None]
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.attn_drop_p if self.training else 0.0)
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            att = att.masked_fill(causal_mask, float("-inf"))
            if self.key_pad_mask is not None:
                att = att.masked_fill(~self.key_pad_mask[:, None, None, :], -1e9)
            att = F.softmax(att, dim=-1)
            att = F.dropout(att, p=self.attn_drop_p, training=self.training)
            out = att @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.wo(out)), new_cache

class TransformerMLP(nn.Module):
    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * mult)
        self.fc2 = nn.Linear(dim * mult, dim)
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = TransformerMLP(dim, dropout=dropout)

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs one Transformer block and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        a, new_cache = self.attn(self.ln1(x), cos, sin, pos_offset, cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, new_cache

class LegalTransformerLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), n_heads (number of heads), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full Legal Transformer model and returns nothing.
    def __init__(self, vocab_size, dim=896, n_layers=24, n_heads=14, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads, dropout) for _ in range(n_layers)])
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:
            self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope_cache(self.head_dim, max_len, device=torch.device("cpu"))
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            nn.init.normal_(blk.attn.wo.weight, mean=0.0, std=0.02 * residual_scale)
            nn.init.normal_(blk.mlp.fc2.weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), kv_cache (saved keys and values) and pos_offset (position offset) as input, runs the model and returns next-token scores and the cache.
    def forward(self, input_ids, kv_cache=None, pos_offset=0):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, lc = blk(x, self.rope_cos, self.rope_sin, pos_offset, layer_cache)
            new_cache.append(lc)
        logits = self.lm_head(self.norm_f(x))
        return logits, new_cache

class LegalTokenizer(LegalByteBPETokenizer):

    # It returns the vocabulary size.
    @property
    def vocab_size(self):
        return len(self.vocab)

    # It returns the padding token id.
    @property
    def pad_id(self):
        return self.pad_token_id

    # It returns the end-of-text token id.
    @property
    def eos_id(self):
        return self.eos_token_id

    bos_id = None

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):
        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            chunk = m.group()
            base = m.start()
            b2c = []
            for ci, ch in enumerate(chunk):
                b2c.extend([base + ci] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[b] for b in chunk.encode("utf-8"))
            pos = 0
            for piece in self._bpe(sym):
                nb = len(piece)
                ids.append(self.vocab[piece])
                offs.append((b2c[pos], b2c[pos + nb - 1] + 1))
                pos += nb
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return LegalTokenizer.load(tok_path)

# It takes path (file path) as input, finds the real checkpoint file to load and returns (file path, temporary folder or None).
def _resolve_ckpt(path):
    if os.path.isfile(path):
        return path, None
    if not os.path.isdir(path):
        raise FileNotFoundError(path)
    root = None
    for dp, _dn, fn in os.walk(path):
        if "data.pkl" in fn:
            root = dp
            break
    if root is None:
        raise FileNotFoundError(f"{path} is a folder without data.pkl")
    prefix = os.path.splitext(os.path.basename(os.path.normpath(root)))[0] or "archive"
    tmpdir = tempfile.mkdtemp(prefix="legal_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for dp, _dn, fn in os.walk(root):
            for f in sorted(fn):
                full = os.path.join(dp, f)
                z.write(full, arcname=prefix + "/" + os.path.relpath(full, root).replace(os.sep, "/"))
    return out, tmpdir

class Backbone:

    # It takes model, cfg (settings) and rope_len (rotary table length) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg, rope_len):
        self.model = model
        self.config = dict(cfg)
        self.d_model = int(cfg["dim"])
        self.max_len = TRAINED_SEQ_TOKENS
        self.rope_len = rope_len
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False
        self.causal = True

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag: bool):
        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional
        for blk in self.model.blocks:
            blk.attn.bidirectional = self.bidirectional

    # It takes input_ids (token ids) and attention_mask (mask of real tokens) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None):
        m = self.model
        B, T = input_ids.shape
        if T > self.rope_len:
            raise ValueError(f"sequence length {T} > RoPE table {self.rope_len}; "
                             f"load_backbone(..., max_len>={T}) (trained only up to {self.max_len})")
        kpm = None
        if attention_mask is not None:
            am = attention_mask.to(input_ids.device).bool()
            if not bool(am.all()):
                kpm = am
        attns = [blk.attn for blk in m.blocks]
        for a in attns:
            a.key_pad_mask = kpm
        try:
            x = m.drop(m.tok_emb(input_ids))
            for blk in m.blocks:
                x, _ = blk(x, m.rope_cos, m.rope_sin, 0, None)
            return m.norm_f(x)
        finally:
            for a in attns:
                a.key_pad_mask = None

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):
        return [n for n, mod in self.model.named_modules()
                if n.startswith("blocks.") and isinstance(mod, nn.Linear)]

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

# It takes ckpt_path (checkpoint path), device, max_len (max length) and config (model settings) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", max_len=None, config=None):
    path, tmp = _resolve_ckpt(ckpt_path)
    try:
        pkg = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    if isinstance(pkg, dict) and "state_dict" in pkg:
        sd, cfg = pkg["state_dict"], dict(pkg.get("config") or {})
    else:
        sd, cfg = pkg, {}
    if config:
        cfg.update(config)
    sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in sd.items()}
    if "vocab_size" not in cfg or "dim" not in cfg:
        w = sd["tok_emb.weight"]
        cfg.setdefault("vocab_size", w.shape[0]); cfg.setdefault("dim", w.shape[1])
        cfg.setdefault("n_layers", 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks.")))
    for k in ("n_heads", "n_layers"):
        if k not in cfg:
            raise ValueError(f"checkpoint config lacks {k!r}; pass config={{'{k}': ...}}")
    cfg.setdefault("dropout", 0.1)
    cfg.setdefault("tied", "lm_head.weight" not in sd or sd["lm_head.weight"].data_ptr() == sd["tok_emb.weight"].data_ptr())
    rope_len = max(int(cfg.get("max_len", NOTEBOOK_ROPE_LEN)), int(max_len or 0))
    model = LegalTransformerLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                               n_heads=cfg["n_heads"], max_len=rope_len,
                               dropout=cfg["dropout"], tie=cfg["tied"])
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if cfg["tied"]:
        missing = [k for k in missing if k != "lm_head.weight"]
    if missing or unexpected:
        print("!" * 70, f"\n!!! STATE_DICT MISMATCH  missing={missing[:10]} (n={len(missing)})  "
              f"unexpected={unexpected[:10]} (n={len(unexpected)})\n" + "!" * 70)
        raise RuntimeError("checkpoint does not match Legal-Transformer architecture")
    if cfg["tied"]:
        assert model.lm_head.weight is model.tok_emb.weight
    model.to(device)
    bb = Backbone(model, cfg, rope_len)
    bb.max_len = min(TRAINED_SEQ_TOKENS, rope_len)
    return bb

Writing arch_legal_transformer.py


In [5]:
# Import code, load data and build training config
PATHS_USED = [CKPT, TOK]

import sys, os, json, time, glob, random, zipfile
import numpy as np, torch
sys.path.insert(0, os.getcwd())
import rr_core as C, rr_peft as P
import arch_legal_transformer as A
import importlib
for _m in (C, P, A): importlib.reload(_m)
DECISIONS = dict(EPOCHS=EPOCHS, BS=BS, ACCUM=ACCUM, W=W, CTX=CTX, HEAD_LR=HEAD_LR, CLASS_POWER=CLASS_POWER, LABEL_SMOOTH=LABEL_SMOOTH, TAUS=TAUS, DEV_SEEDS=DEV_SEEDS, FULL_SEEDS=FULL_SEEDS, SWA_LAST=SWA_LAST, GRU_HIDDEN=GRU_HIDDEN, RARE_DUP=RARE_DUP, RARE_SHARE=RARE_SHARE, TOK_MASK=TOK_MASK, SNAP_EPOCHS=SNAP_EPOCHS, VIT_LAMS=VIT_LAMS, POS_FEAT=POS_FEAT, NEVER_PREDICT_BELOW=NEVER_PREDICT_BELOW, PEFT_LR=PEFT_LR, R=R, ALPHA=ALPHA, LORA_DROPOUT=LORA_DROPOUT, HEAD_DROPOUT=HEAD_DROPOUT)
missing = [k for k, v in DECISIONS.items() if v is None]
assert not missing, "Fill these in the Settings cell first: " + ", ".join(missing)
for p in PATHS_USED: assert p and os.path.exists(p), f"Path not found: {p!r} - attach the file and paste its path in the Settings cell"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, "| GPUs:", torch.cuda.device_count())
os.makedirs(OUT, exist_ok=True); WORK = os.path.join(OUT, "_work"); os.makedirs(WORK, exist_ok=True)
RESUME_DIRS = [d for d in glob.glob("/kaggle/input/**/_work", recursive=True)]

C.ensure_phase3_data(repo=HF_DATASET)
raw = C.load_all(); labels = raw["labels"]; K = len(labels); label2i = {l: i for i, l in enumerate(labels)}
if SMOKE:
    for part in ("train", "dev", "test"): raw[part] = dict(list(raw[part].items())[:int(SMOKE_DOCS)])
print("labels:", labels)
for part in ("train", "dev", "test"):
    ds = raw[part]; print(f"{part:5s}: {len(ds)} documents ({sum(d['domain']=='CL' for d in ds.values())} CL / {sum(d['domain']=='IT' for d in ds.values())} IT), {sum(len(d['sents']) for d in ds.values())} sentences")
ids = [d["id"] for d in raw["test"].values()]
assert len(ids) == len(set(ids)), "a document id occurs in both CL_test and IT_test - read the evaluation script to see how the two files must be merged"
tr_lab = [l for d in raw["train"].values() for l in d["labels"]]
cnt = {l: tr_lab.count(l) for l in labels}; print("train label counts:", cnt)

tok = A.load_tokenizer(TOK)
bb0 = A.load_backbone(CKPT, device="cpu", max_len=1024)
MAXTOK = 640
print("backbone parameters:", f"{bb0.n_params:,}", "| usable context:", MAXTOK, "tokens")
assert CTX == "sentence" or W + 2 <= MAXTOK, f"W={W} is longer than what this model was trained for ({MAXTOK} tokens)"
MAX_SENT = 200 if CTX == "window" else min(200, MAXTOK - 2)
data = dict(labels=labels, pad_id=tok.pad_id, raw=raw)
for part in ("train", "dev", "test"): data[part] = C.encode_docs(raw[part], tok, label2i, MAX_SENT) if part != "test" else C.encode_docs(raw[part], tok, {**label2i}, MAX_SENT)
lens = [sum(len(s) for s in e.ids) for e in data["train"].values()]
print("tokens per training document: mean", int(np.mean(lens)), "max", max(lens))
CFG = dict(epochs=EPOCHS, bs=BS, accum=ACCUM, W=W, ctx=CTX, class_power=CLASS_POWER, label_smooth=LABEL_SMOOTH, swa_last=SWA_LAST, rare_dup=RARE_DUP, rare_share=RARE_SHARE, tok_mask=TOK_MASK, snap_epochs=SNAP_EPOCHS, vit_lams=VIT_LAMS, never_predict_below=NEVER_PREDICT_BELOW, min_count=max(1, NEVER_PREDICT_BELOW), warmup=0.06, wd=0.01, clip=1.0, save_every=1500, emit_zips=False, tta_step=0.3)

device: cuda | GPUs: 2


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  downloaded CL_train.json -> /kaggle/working/phase3_data/Phase_3_RR/Train/CL_train.json


CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

  downloaded CL_dev.json -> /kaggle/working/phase3_data/Phase_3_RR/Train/CL_dev.json


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

... (5 outputs left out)

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

  downloaded CL_test.json -> /kaggle/working/phase3_data/Phase_3_RR/Test/CL_test.json


IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

  downloaded IT_test.json -> /kaggle/working/phase3_data/Phase_3_RR/Test/IT_test.json
labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
train: 80 documents (40 CL / 40 IT), 16845 sentences
dev  : 10 documents (5 CL / 5 IT), 2142 sentences
test : 10 documents (5 CL / 5 IT), 2197 sentences
train label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
backbone parameters: 271,180,672 | usable context: 640 tokens
tokens per training document: mean 9585 max 84245


In [6]:
# Build the model with LoRMA
NAME = "rr1_lt_lorma"
# It takes seed (random seed) as input, builds the model with the chosen fine-tuning method and its learning rates and returns (model, learning-rate groups, info).
def build(seed):
    bb = A.load_backbone(CKPT, device="cpu", max_len=1024)
    n = P.apply_linear_wrappers(bb, "lorma", bb.linear_names(), r=R, alpha=ALPHA, dropout=LORA_DROPOUT)
    model = C.RRModel(bb, K, CTX, GRU_HIDDEN, HEAD_DROPOUT, n_extra=4 if POS_FEAT else 0)
    return model, C.make_groups(model, HEAD_LR, PEFT_LR), f"lorma: {n} linear layers wrapped, r={R}, alpha={ALPHA}"

In [7]:
# Run training and evaluation
t_start = time.time()
summary = C.run_experiment(NAME, build, CFG, data, OUT, WORK, resume_dirs=RESUME_DIRS, taus=TAUS, dev_seeds=DEV_SEEDS, full_seeds=FULL_SEEDS, stages=STAGES)
print("total time", C._hms(time.time() - t_start))

labels that are never predicted (fewer than 50 training sentences): ['PrecedentOverruled']

===== rr1_lt_lorma: stage 'dev' (train -> dev), seeds [0] =====
  seed 0: trainable parameters 10,172,493 of 281,353,165 (3.62 %)  lorma: 162 linear layers wrapped, r=16, alpha=32
  micro-batches/epoch 965 (bs 4 x accum 4), optimizer steps/epoch 241, total 1446; rare labels [1, 5, 7, 8, 11, 12]


epoch 1/6:   0%|          | 0/241 [00:00<?, ?it/s]

  epoch 1  train_ce 1.8562  secs 422.7298  val_macro_F1 0.4350  val_CL 0.3574  val_IT 0.4968  | epoch 1/6, run ETA 0h37m  [overall 4% | elapsed 0h07m | ETA 3h09m]


epoch 2/6:   0%|          | 0/241 [00:00<?, ?it/s]

  epoch 2  train_ce 1.3170  secs 418.8317  val_macro_F1 0.4475  val_CL 0.3565  val_IT 0.4862  | epoch 2/6, run ETA 0h29m  [overall 8% | elapsed 0h14m | ETA 3h00m]


epoch 3/6:   0%|          | 0/242 [00:00<?, ?it/s]

... (37 outputs left out)

epoch 5/6:   0%|          | 0/272 [00:00<?, ?it/s]

  epoch 5  train_ce 0.8739  secs 474.6520  | epoch 5/6, run ETA 0h07m  [overall 96% | elapsed 3h08m | ETA 0h08m]


epoch 6/6:   0%|          | 0/272 [00:00<?, ?it/s]

  epoch 6  train_ce 0.8326  secs 474.5508  | epoch 6/6, run ETA 0h00m  [overall 100% | elapsed 3h16m | ETA 0h00m]
  seed 2 finished: probabilities of 3 seed(s) so far are saved
total time 3h19m


In [11]:
# Show dev scores of all variants
rows = []
for stg, info in summary["stages"].items():
    for key, sc in info.get("scores", {}).items():
        rows.append((sc["all"]["macro_f1"], key, {d: round(v["macro_f1"], 4) for d, v in sc.items()}))
for f1, key, per in sorted(rows, reverse=True): print(f"dev Macro-F1 {f1:.4f}   {key:14s}  {per}")
print("\nSubmission zips (each contains only prediction.json at the zip root):")
for f in sorted(glob.glob(os.path.join(OUT, "*_submission.zip"))): print("  ", os.path.basename(f), f"{os.path.getsize(f)/1e6:.1f} MB")
print("\nThe '__full_' zips come from the model retrained on ALL documents (train + dev): these are the ones to submit. "
      "Pick the tau (and weights kind) that was best on the 'dev' rows; the dev rows themselves were produced without ever seeing the dev documents.")

dev Macro-F1 0.4712   final_tau0      {'all': 0.4712, 'CL': 0.3884, 'IT': 0.5309}
dev Macro-F1 0.4712   best_tau0       {'all': 0.4712, 'CL': 0.3884, 'IT': 0.5309}
dev Macro-F1 0.4709   avg_tau0        {'all': 0.4709, 'CL': 0.3854, 'IT': 0.5337}
dev Macro-F1 0.4705   ep5_tau0        {'all': 0.4705, 'CL': 0.3848, 'IT': 0.5335}
dev Macro-F1 0.4675   snapens_tau0    {'all': 0.4675, 'CL': 0.381, 'IT': 0.5325}
dev Macro-F1 0.4552   ep4_tau0        {'all': 0.4552, 'CL': 0.3741, 'IT': 0.5122}
dev Macro-F1 0.4479   ep3_tau0        {'all': 0.4479, 'CL': 0.3614, 'IT': 0.5046}

Submission zips (each contains only prediction.json at the zip root):

The '__full_' zips come from the model retrained on ALL documents (train + dev): these are the ones to submit. Pick the tau (and weights kind) that was best on the 'dev' rows; the dev rows themselves were produced without ever seeing the dev documents.


In [ ]:
# Pick best decoding and write top 5 submissions
import re, csv, shutil, itertools
TOP_N = 5
SELECT_BY = "robust"
DEVSTAGE_W = 0.3
NEVER_BELOW_ = 50
TAUS_ = [0.0, 0.1, 0.2, 0.3, 0.4]; LAMS_ = [0.0, 0.5, 1.0, 1.5, 2.0]; PMIS_ = [0.0, 0.5, 1.0]
labels = raw["labels"]; K = len(labels); l2i = {l: i for i, l in enumerate(labels)}
mk = lambda part: {k: C.Enc(k, d["id"], d["domain"], [[]] * len(d["sents"]), (np.array([l2i[x] for x in d["labels"]]) if d["labels"] else None)) for k, d in raw[part].items()}
tr_e, dv_e, te_e = mk("train"), mk("dev"), mk("test")
_, prior = C.class_weights({**tr_e, **dv_e}, K, min_count=NEVER_BELOW_)
cnt_tr, cnt_all = C.label_counts(tr_e, K), C.label_counts({**tr_e, **dv_e}, K)
ban = {"dev": tuple(int(i) for i in np.where(cnt_tr < NEVER_BELOW_)[0]), "full": tuple(int(i) for i in np.where(cnt_all < NEVER_BELOW_)[0])}
logT0 = {"dev": C.transitions(tr_e, K), "full": C.transitions({**tr_e, **dv_e}, K)}
logT = lambda stage, pmi: logT0[stage] - pmi * np.log(prior)[None, :]
pf = lambda stage, kind, part: os.path.join(OUT, f"{NAME}_{stage}_{kind}_{part}_probs.npz")
kinds_dev = sorted(k for k in ("best", "final", "avg", "snapens") + tuple(f"ep{e}" for e in range(1, 60)) if os.path.exists(pf("dev", k, "dev")))
cand_kinds = [k for k in kinds_dev if os.path.exists(pf("full", k, "test")) or (k == "best" and os.path.exists(pf("dev", k, "test")))]
print("weights available:", kinds_dev, "-> candidates:", cand_kinds)
dev_p = {k: C.load_probs(pf("dev", k, "dev")) for k in kinds_dev}
# It takes k (setting key) as input, loads test probabilities, mixing the two training stages and returns test probabilities per document.
def test_p(k):
    if not os.path.exists(pf("full", k, "test")): return C.load_probs(pf("dev", k, "test"))
    full = C.load_probs(pf("full", k, "test"))
    if DEVSTAGE_W > 0 and os.path.exists(pf("dev", k, "test")):
        ds = C.load_probs(pf("dev", k, "test")); return {d: (full[d] + DEVSTAGE_W * ds[d]) / (1 + DEVSTAGE_W) for d in full}
    return full
SETTINGS = [(t, l, p, b) for t in TAUS_ for l in LAMS_ for p in (PMIS_ if l else [0.0]) for b in ((False, True) if ban["dev"] else (False,))]
G = {k: {s: C.score_split(dv_e, dev_p[k], K, prior, s[0], logT=logT("dev", s[2]), lam=s[1], banned=ban["dev"] if s[3] else ()) for s in SETTINGS} for k in cand_kinds}
f1 = lambda sc: sc["all"]["macro_f1"]; sname = lambda s: f"tau {s[0]:g} lambda {s[1]:g} pmi {s[2]:g} ban {s[3]}"
robust = {s: float(np.mean([f1(G[k][s]) for k in cand_kinds])) for s in SETTINGS}
plain = (0.0, 0.0, 0.0, False)
print("\nDev Macro-F1 (labels that occur in dev) of plain decoding per weights:  " + "  ".join(f"{k}: {f1(G[k][plain]):.4f}" for k in cand_kinds))
order = sorted(SETTINGS, key=lambda s: -robust[s])
print("Best settings by the MEAN over all weights (the number to trust):")
for s in order[:8]: print(f"  mean {robust[s]:.4f} | " + " ".join(f"{k} {f1(G[k][s]):.3f}" for k in cand_kinds) + f" | {sname(s)}")
cands = [dict(score=0.5 * (f1(G[k][s]) + robust[s]), dev=f1(G[k][s]), kind=k, s=s) for k in cand_kinds for s in SETTINGS]
top = []
for c in sorted(cands, key=lambda c: -(c["dev"] if SELECT_BY == "dev" else c["score"])):
    if any(t["kind"] == c["kind"] for t in top[:]) and sum(t["kind"] == c["kind"] for t in top) >= 1 and len(cand_kinds) >= TOP_N: continue
    if sum(t["s"] == c["s"] for t in top) >= 2: continue
    top.append(c)
    if len(top) == TOP_N: break
for c in sorted(cands, key=lambda c: -c["score"]):
    if len(top) == TOP_N: break
    if c not in top: top.append(c)
D = os.path.join(OUT, f"TOP{TOP_N}"); shutil.rmtree(D, ignore_errors=True); os.makedirs(D)
print(f"\nThe top {TOP_N} (dev numbers are only a guide; 'best' = train-only model at its best dev epoch):")
for r, c in enumerate(top, 1):
    t, l, p, b = c["s"]; ptest = test_p(c["kind"])
    pred = C.predictions_for(te_e, ptest, labels, prior, t, logT("full", p), l, ban["full"] if b else ()); C.check_prediction(pred, raw["test"], labels)
    nm = f"{r}_lt_{c['kind']}_t{t:g}_v{l:g}_p{p:g}{'_ban' if b else ''}.zip"; assert len(nm) <= 64, nm; C.write_submission(pred, os.path.join(D, nm))
    print(f"  {nm:40s} dev F1 {c['dev']:.4f} (rank score {c['score']:.4f}) | {sname(c['s'])}")
with zipfile.ZipFile(os.path.join(OUT, f"TOP{TOP_N}_submissions.zip"), "w") as z:
    for nm in sorted(os.listdir(D)): z.write(os.path.join(D, nm), nm)
print(f"\nDownload TOP{TOP_N}/ (or the bundle; unzip it first). Each zip is a ready Codabench submission.")
c = top[0]; sc = G[c["kind"]][c["s"]]["all"]; pl = sc["per_label"]
print(f"\n#1: weights {c['kind']}, {sname(c['s'])}: dev Macro-F1 {sc['macro_f1']:.4f}, precision {sc['macro_p']:.3f}, recall {sc['macro_r']:.3f}")
print(f"{'label':24s}{'P':>7}{'R':>7}{'F1':>7}{'n':>7}")
for i, l in enumerate(labels): print(f"{l:24s}{pl['P'][i]:7.2f}{pl['R'][i]:7.2f}{pl['F1'][i]:7.2f}{int(pl['support'][i]):7d}")
cm = np.array(sc["confusion"]); off = sorted(((cm[i, j], labels[i], labels[j]) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0), reverse=True)[:8]
print("\nBiggest confusions (gold -> predicted):  " + "; ".join(f"{a} -> {b} ({n})" for n, a, b in off))

weights available: ['avg', 'best', 'ep3', 'ep4', 'ep5', 'final', 'snapens'] -> candidates: ['avg', 'best', 'ep3', 'ep4', 'ep5', 'final', 'snapens']


In [ ]:
# Zip all outputs
import shutil
zp = os.path.join(OUT, "RR1_LegalTransformer_LoRMA_ALL_outputs.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(OUT, "*"))):
        if os.path.isfile(f) and not f.endswith("_ALL_outputs.zip"): z.write(f, os.path.basename(f))
    for f in glob.glob(os.path.join(WORK, "*.npz")): z.write(f, "_work/" + os.path.basename(f))
print(zp, f"{os.path.getsize(zp)/1e6:.1f} MB")